In [1]:
from datasets import DatasetDict, load_dataset
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from torchcrf import CRF

In [2]:
EMBED_DIM = 300
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3

In [3]:
glove_path = "data/dolma_300_2024_1.2M.100_combined.txt"

embeddings_dict = {}

with open(file = glove_path, mode = 'r', encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], "float32")
        embeddings_dict[word] = vector

In [4]:
BASE = "https://huggingface.co/datasets/conll2003/resolve/refs%2Fconvert%2Fparquet/conll2003"

dataset = DatasetDict({
    split: load_dataset("parquet", data_files={split: f"{BASE}/{split}/0000.parquet"}, split=split)
    for split in ["train", "validation", "test"]
})

In [5]:
OOV_VECTOR = np.zeros(300, dtype="float32")
OOV_VECTOR.flags.writeable = False

class NERDataset(Dataset):
    def __init__(self, tokens, tags, embed_map):
        self.samples = [(t, g) for t, g in zip(tokens, tags) if len(t)>0]
        self.embed_map = embed_map

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        toks, tags = self.samples[idx]
        vecs = torch.stack([
            torch.from_numpy(self.embed_map.get(t, self.embed_map.get(t.lower(), OOV_VECTOR)))
            for t in toks
        ])
        return vecs, torch.tensor(tags, dtype=torch.long)
    
def collate_fn(batch):
    seqs, tags = zip(*batch)
    X = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    y = pad_sequence(tags, batch_first=True, padding_value=-1)
    mask = (y != -1).bool()
    return X, y, mask

In [6]:
class LinearChainCRF(nn.Module):
    def __init__(self, embed_dim, num_tags):
        super().__init__()
        self.linear = nn.Linear(embed_dim, num_tags)
        self.crf = CRF(num_tags=num_tags, batch_first=True)

    def forward(self, x, tags, mask):
        emissions = self.linear(x)
        return -self.crf(emissions, tags, mask=mask)

    def decode(self, x, mask):
        emissions = self.linear(x)
        return self.crf.decode(emissions, mask=mask)

## Training

In [7]:
num_tags = dataset['train'].features['ner_tags'].feature.num_classes
model = LinearChainCRF(EMBED_DIM, num_tags)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_loader = DataLoader(
    NERDataset(dataset["train"]["tokens"], dataset["train"]["ner_tags"], embeddings_dict),
    batch_size=32, shuffle=True, collate_fn=collate_fn
)

model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    for X, y, mask in train_loader:
        optimizer.zero_grad()
        loss = model(X, y, mask) / mask.sum()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}  loss={total_loss:.4f}")


/var/folders/sx/dymxkj2j1636xvkc7jyhyp840000gn/T/ipykernel_88337/3806392515.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/ec2-user/croot/libtorch_1771497326510/work/torch/csrc/utils/tensor_numpy.cpp:209.)
  torch.from_numpy(self.embed_map.get(t, self.embed_map.get(t.lower(), OOV_VECTOR)))


Epoch 1  loss=239.2404
Epoch 2  loss=142.1065
Epoch 3  loss=116.8955
Epoch 4  loss=101.3481
Epoch 5  loss=91.1784
Epoch 6  loss=84.6036
Epoch 7  loss=79.9886
Epoch 8  loss=76.3422
Epoch 9  loss=73.9550
Epoch 10  loss=71.9592
Epoch 11  loss=70.3800
Epoch 12  loss=69.1872
Epoch 13  loss=68.0979
Epoch 14  loss=67.1545
Epoch 15  loss=66.5278
Epoch 16  loss=66.1922
Epoch 17  loss=65.4751
Epoch 18  loss=64.9852
Epoch 19  loss=64.4634
Epoch 20  loss=64.2143


In [8]:
ner_labels = dataset['train'].features['ner_tags'].feature.names

val_loader = DataLoader(
    NERDataset(dataset["validation"]["tokens"], dataset["validation"]["ner_tags"], embeddings_dict),
    batch_size=32, shuffle=False, collate_fn=collate_fn
)

model.eval()
all_preds, all_golds = [], []

with torch.no_grad():
    for X, y, mask in val_loader:
        batch_preds = model.decode(X, mask)
        for pred_seq, gold_seq, m in zip(batch_preds, y, mask):
            valid_len = m.sum().item()
            all_preds.append([ner_labels[i] for i in pred_seq[:valid_len]])
            all_golds.append([ner_labels[i] for i in gold_seq[:valid_len].tolist()])
            

In [17]:
from seqeval.metrics import classification_report
from seqeval.metrics import f1_score

print(f"F1-score: {f1_score(all_golds, all_preds):.2f}")
print(classification_report(all_golds, all_preds))


F1-score: 0.75
              precision    recall  f1-score   support

         LOC       0.83      0.81      0.82      1837
        MISC       0.76      0.61      0.68       922
         ORG       0.66      0.52      0.58      1341
         PER       0.86      0.81      0.83      1842

   micro avg       0.79      0.71      0.75      5942
   macro avg       0.78      0.69      0.73      5942
weighted avg       0.79      0.71      0.75      5942

